# SQL Fundamentals Workshop — Week 1 Quiz

**Prof. Alex Labrinidis · March 25, 2026**



Practice queries on the NYC Rolling Sales dataset.

Each question asks you to write a SQL query that answers a real estate analytics question.



---

## Setup



Run the cell below before attempting any questions.

It loads the NYC sales data and defines the `run()` helper function.


In [11]:
import sqlite3
import pandas as pd

pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_URL = 'https://db.cs.pitt.edu/data/nyc_sales_20260303.csv'

df_raw = pd.read_csv(DATA_URL)
conn = sqlite3.connect(':memory:')
df_raw.to_sql('nyc_sales', conn, if_exists='replace', index=False)

print(f'Loaded {len(df_raw):,} rows into nyc_sales table.')

def run(sql):
    """Run a SELECT query and return results as a DataFrame."""
    return pd.read_sql_query(sql, conn)

Loaded 79,335 rows into nyc_sales table.


---

## Schema reference



```

nyc_sales(

  borough                   INTEGER,  -- 1=Manhattan 2=Bronx 3=Brooklyn 4=Queens 5=Staten Island

  neighborhood              TEXT,

  building_class_category   TEXT,

  address                   TEXT,

  zip_code                  TEXT,

  residential_units         INTEGER,

  commercial_units          INTEGER,

  total_units               INTEGER,

  land_square_feet          INTEGER,

  gross_square_feet         INTEGER,

  year_built                INTEGER,

  sale_price                INTEGER,

  sale_date                 TEXT      -- format: YYYY-MM-DD

)

```


---

## Q1 — Filtering + sorting + DISTINCT



List the distinct zip codes in Queens (borough = 4) that had at least one sale above $750,000. Sort zip codes alphabetically.

In [12]:
# Write your SQL query here
run('''
SELECT DISTINCT zip_code
FROM nyc_sales
WHERE borough = 4
  AND sale_price > 750000
ORDER BY zip_code;
''')

,zip_code
0,NaN
1,"11,001.00"
2,"11,004.00"
3,"11,005.00"
4,"11,040.00"
...,...
59,"11,691.00"
60,"11,692.00"
61,"11,693.00"
62,"11,694.00"


## Q2 — NULL-aware counting

How many sales in the Bronx (borough = 2) have a recorded `gross_square_feet` value greater than zero?

In [13]:
# Write your SQL query here
run('''
SELECT COUNT(*) AS bronx_sales_with_valid_gross_sqft
FROM nyc_sales
WHERE borough = 2
  AND gross_square_feet IS NOT NULL
  AND gross_square_feet > 0;
''')

,bronx_sales_with_valid_gross_sqft
0,4374


## Q3 — Filtering with multiple conditions



List all sales in Staten Island (borough = 5) with a valid sale price greater than $500,000. Show the address, neighborhood, sale price, and sale date. Sort by sale price descending.

In [14]:
# Write your SQL query here
run('''
SELECT address, neighborhood, sale_price, sale_date
FROM nyc_sales
WHERE borough = 5
  AND sale_price IS NOT NULL
  AND sale_price > 500000
ORDER BY sale_price DESC;
''')

,address,neighborhood,sale_price,sale_date
0,2975 RICHMOND AVENUE,LA TOURETTE PARK,37605875,2025-05-22
1,175 LAKE AVENUE,MARINERS HARBOR,16750000,2025-03-20
2,111 QUINTARD STREET,SOUTH BEACH,11721900,2025-11-07
3,1125 BAY STREET,ROSEBANK,10500000,2025-11-24
4,1590 HYLAN BOULEVARD,SOUTH BEACH,9215800,2025-11-07
...,...,...,...,...
3368,105 CLARK LANE,TOMPKINSVILLE,505000,2025-02-14
3369,34 JOURNEAY STREET,MARINERS HARBOR,504700,2025-10-31
3370,614 ILYSSA WAY,ARDEN HEIGHTS,503500,2025-08-20
3371,123 PENDLETON PLACE,NEW BRIGHTON,503500,2025-11-20


## Q4 — Text cleaning + counting



How many distinct building class categories contain the word `CONDO`? Apply proper text cleaning before comparing.

In [15]:
# Write your SQL query here
run('''
SELECT COUNT(DISTINCT TRIM(UPPER(building_class_category))) AS condo_category_count
FROM nyc_sales
WHERE TRIM(UPPER(building_class_category)) LIKE '%CONDO%';
''')

,condo_category_count
0,16


## Q5 — Aggregates with a filter



What is the total number of sales, total dollar volume, and average sale price in Brooklyn (borough = 3) — considering only sales with a valid price greater than zero?

In [16]:
# Write your SQL query here
run('''
SELECT
    COUNT(*) AS total_sales,
    SUM(sale_price) AS total_dollar_volume,
    AVG(sale_price) AS average_sale_price
FROM nyc_sales
WHERE borough = 3
  AND sale_price > 0;
''')

,total_sales,total_dollar_volume,average_sale_price
0,13641,23976342332,"1,757,667.50"


## Q6 — LIKE with multiple conditions



Return all distinct addresses in Manhattan (borough = 1) that contain the street name `BROADWAY` and were sold for more than $1 million. Sort by sale price descending.

In [17]:
# Write your SQL query here
run('''
SELECT DISTINCT address, sale_price
FROM nyc_sales
WHERE borough = 1
  AND UPPER(address) LIKE '%BROADWAY%'
  AND sale_price > 1000000
ORDER BY sale_price DESC;
''')

,address,sale_price
0,529 BROADWAY,213000000
1,356-360 WEST BROADWAY,87500000
2,1370 BROADWAY,75250000
3,360 BROADWAY,57575000
4,"640 BROADWAY, UNIT2",49500000
...,...,...
245,"2250 BROADWAY, 3E",1070000
246,"176 BROADWAY, 9F",1060000
247,"692 BROADWAY, 607",1060000
248,"268 EAST BROADWAY, A1104",1040000


## Q7 — Single aggregate



What is the average sale price in Brooklyn (borough = 3)? Exclude $0 sales from the calculation.

In [18]:
# Write your SQL query here
run('''
SELECT AVG(sale_price) AS brooklyn_average_sale_price
FROM nyc_sales
WHERE borough = 3
  AND sale_price > 0;
''')

,brooklyn_average_sale_price
0,"1,757,667.50"


## Q8 — Scalar subquery with data quality awareness



Which sales in Brooklyn (borough = 3) had a sale price more than 3 times the Brooklyn average sale price? Show the address, neighborhood, and sale price. Exclude $0 sales from both the main query and the average calculation. You can reuse the query from Q7 as a subquery — but do not hard-code the number you got as the answer.

In [19]:
# Write your SQL query here
run('''
SELECT address, neighborhood, sale_price
FROM nyc_sales
WHERE borough = 3
  AND sale_price > 0
  AND sale_price > (
      SELECT 3 * AVG(sale_price)
      FROM nyc_sales
      WHERE borough = 3
        AND sale_price > 0
  );
''')

,address,neighborhood,sale_price
0,1756 86 STREET,BATH BEACH,8300000
1,8622 BAY PARKWAY,BATH BEACH,8500000
2,8053 HARBOR VIEW TERRACE,BAY RIDGE,5850000
3,230 73RD STREET,BAY RIDGE,7125000
4,7410 RIDGE BLVD,BAY RIDGE,24375000
...,...,...,...
548,200 SOUTH 3RD STREET,WILLIAMSBURG-SOUTH,9573800
549,96 SOUTH 9 STREET,WILLIAMSBURG-SOUTH,6100000
550,"146 SOUTH 4 STREET, 1",WILLIAMSBURG-SOUTH,9900000
551,20 TERRACE PLACE,WINDSOR TERRACE,6900000


## Q9 — Multi-condition data quality summary



Build a data quality summary for the `year_built` column: count the total rows, how many have a NULL `year_built`, how many have a `year_built` of 0, and how many have a plausible value (built after 1600 and no later than 2026). Label each column clearly.

In [20]:
# Write your SQL query here
run('''
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN year_built IS NULL THEN 1 END) AS null_year_built,
    COUNT(CASE WHEN year_built = 0 THEN 1 END) AS zero_year_built,
    COUNT(CASE
              WHEN year_built > 1600 AND year_built <= 2026
              THEN 1
          END) AS plausible_year_built
FROM nyc_sales;
''')

,total_rows,null_year_built,zero_year_built,plausible_year_built
0,79335,5228,0,74106


---

_End of Week 1 Quiz_